# A bússola de vocabulário do PBIA — UE, China, EUA e OCDE como quatro polos

Este notebook constrói um **gráfico de posicionamento de quatro polos** (uma
"bússola de vocabulário") que localiza o **PBIA** em relação a cinco planos
internacionais de IA — **AI Continent Action Plan** (UE), **Apply AI
Strategy** (UE), **China New Gen** (China, 2017), **AI+ / AI Plus** (China,
2023) e **America's AI Action Plan** (EUA) — usando a **Recomendação da OCDE**
apenas para definir o polo negativo do eixo Y (não como um documento
posicionado em pé de igualdade com os demais).

**Definição dos eixos:**

- **Eixo X positivo** — vocabulário mais específico do bloco **UE** (extraído
  de `AI_CONTINENT_ACTION_PLAN.json` + `Apply_AI_Strategy.json`).
- **Eixo X negativo** — vocabulário mais específico do bloco **China**
  (extraído de `China_New_Gen.json` + `AI_PLUS.json`).
- **Eixo Y positivo** — vocabulário mais específico dos **EUA**
  (`AMERICA_AI_ACTION_PLAN.json`).
- **Eixo Y negativo** — vocabulário mais específico da **OCDE**
  (`OCDE.json`), usado propositalmente como contraste tanto ao eixo X quanto
  ao eixo Y positivo.

O **PBIA** (`PBIA.json`) é o centro da análise: não define nenhum polo — é
posicionado *depois* de os quatro polos já estarem fixados, com base em
quanto do vocabulário de cada polo ele efetivamente usa.

## Reaproveitamento metodológico deste projeto

A extração de texto por JSON, a tokenização, o stemmer (dicionário
curado + sufixos genéricos) e o teste de *keyness* por log-likelihood (G²)
são **reaproveitados integralmente de
[`Estudo Gráfico.ipynb`](./Estudo%20Gr%C3%A1fico.ipynb)**, onde já foram
validados e documentados (inclusive com teste de robustez metodológica na
Seção 4 daquele notebook). Este notebook **não reabre** essas escolhas — ele
as usa como base fixa e constrói, em cima delas, uma pergunta nova: em vez de
um mapa de similaridade livre (MDS, Seção 7 de `Estudo Gráfico.ipynb`), aqui
os quatro polos são **fixados a priori pelos blocos geopolíticos**, e cada
documento (incluindo os que definem os próprios polos) é posicionado pela
mesma régua — o que permite ler diretamente "o quanto este documento soa como
UE vs. como China" e "o quanto soa como EUA vs. como OCDE", em vez de uma
configuração geométrica relativa entre todos os pares.

## Padrão de rigor exigido aqui

- Toda afirmação numérica no texto vem de uma célula de código executada
  nesta sessão.
- Todo parâmetro de decisão (piso mínimo de frequência, tamanho do
  vocabulário por polo, limiar de significância) é explícito no código e
  justificado no texto, com teste de sensibilidade quando aplicável.
- A **desproporção de tamanho entre os documentos e entre os polos** é
  medida explicitamente (Seção 3) e todo escore de vocabulário usado no
  gráfico é uma **frequência relativa** (por 1.000 tokens), nunca contagem
  bruta — exatamente para compensar essa desproporção.
- Nenhum termo autorreferente (nomes de países/blocos/instituições) entra
  no vocabulário dos polos — ver `SELF_REFERENTIAL` na Seção 4.
- Um problema de **reprodutibilidade** foi identificado e corrigido durante a
  construção deste notebook: o desempate de termos com G² exatamente igual
  dependia da ordem de iteração de um `set` do Python, que é aleatorizada
  por processo (hash de strings). Toda função de ordenação abaixo usa um
  **desempate alfabético explícito**, tornando os resultados idênticos entre
  execuções — ver Seção 4.

---
# 1. Extração de texto por documento

In [ ]:
import json
import re
import math
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

BASE = Path(".")

STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "they", "them", "there",
    "etc",
}
TOKEN_RE = re.compile(r"[a-zA-Z]+(?:-[a-zA-Z]+)*")


def join_parts(parts):
    # junta ignorando None (alguns campos de origem são opcionais)
    return " ".join(p for p in parts if p)


def all_strings(obj):
    # extrai recursivamente todo valor string de um dict/list aninhado
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, dict):
        for v in obj.values():
            yield from all_strings(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from all_strings(item)

Sete extratores, um por documento — cada JSON tem um esquema diferente
(`sections[]` simples, `pillars[].sections[]`, árvore recursiva
`sections -> subsections -> items -> boxes`, etc.), então cada um precisa do
seu próprio percurso pelo esquema. Idênticos aos de `Estudo Gráfico.ipynb`
(Seção 1 daquele notebook), reproduzidos aqui para que este notebook rode de
forma independente.

In [ ]:
def extract_pbia(data):
    parts = [data["document"]["title"]]
    parts += data["pbia_objectives"]
    parts += list(all_strings(data["responsible_ai_principles"]))
    parts += list(all_strings(data["pillars"]))
    parts += list(all_strings(data["premises"]))
    parts += list(all_strings(data["windows_of_opportunity"]))
    parts += [t["era"] for t in data["ai_timeline"]]
    parts += [t["event"] for t in data["ai_timeline"]]
    parts.append(data["investments"]["note"])
    parts += data["immediate_impact_sectors"]
    parts += list(all_strings(data["immediate_impact_actions"]))
    parts += list(all_strings(data["structuring_axes"]))
    parts += list(all_strings(data["structuring_actions"]))
    parts += [g["description"] for g in data["governance_examples"]]
    for s in data["sections"]:
        parts.append(s["category"])
        if s.get("subtitle"):
            parts.append(s["subtitle"])
        parts.append(s["text"])
    return join_parts(parts)


def extract_flat_sections(data):
    # documentos cujo texto está numa lista simples sections[] com campo text
    parts = [data["document"]["title"]]
    for s in data["sections"]:
        parts.append(s["category"])
        if s.get("subtitle"):
            parts.append(s["subtitle"])
        parts.append(s["text"])
    return parts


def extract_ocde(data):
    return join_parts(extract_flat_sections(data))


def extract_aiplus(data):
    parts = extract_flat_sections(data)
    parts += data["key_fields"]
    for t in data["development_targets"]:
        parts.append(t["comparison"])
        parts += t["milestones"]
    for k in data["key_initiatives"]:
        parts.append(k["title"])
        parts += k["items"]
    parts += [b["title"] for b in data["basic_support_capabilities"]]
    return join_parts(parts)


def extract_china_new_gen(data):
    # texto em árvore (sections -> subsections -> items -> boxes), cada nível
    # com title/paragraphs -- requer travessia recursiva própria
    parts = [data["document"]["title"], data["document"]["full_title"], data["document"].get("preamble", "")]

    def walk(node, parts):
        if "title" in node:
            parts.append(node["title"])
        if "paragraphs" in node:
            parts.extend(node["paragraphs"])
        for key in ("subsections", "items", "boxes"):
            for child in node.get(key, []):
                walk(child, parts)

    for chapter in data["sections"]:
        walk(chapter, parts)
    return join_parts(parts)


def extract_apply_ai_strategy(data):
    parts = extract_flat_sections(data)
    parts.append(data["document"]["full_title"])
    for c in data["governance_mechanism"]["components"]:
        parts.append(c["name"])
        parts.append(c["role"])
    parts += [s["sector"] for s in data["sectoral_flagships_summary"]]
    return join_parts(parts)


def extract_ai_continent(data):
    parts = extract_flat_sections(data)
    parts.append(data["document"]["full_title"])
    parts += data["document"]["five_key_domains"]
    parts += data["document"]["main_sections"]
    for cs in data["case_studies"]:
        parts.append(cs["title"])
        parts.append(cs["text"])
    for a in data["key_actions"]:
        parts.append(a["action"])
        parts.append(a["section"])
    return join_parts(parts)


def extract_america(data):
    meta = data["document_metadata"]
    parts = [meta["title"], meta["subtitle"], meta["opening_quote"]["text"]]
    intro = data["introduction"]
    parts.append(intro["summary"])
    parts += intro["key_points"]
    parts += intro["cross_cutting_principles"]
    parts.append(intro["referenced_executive_order"]["title"])
    for pillar in data["pillars"]:
        parts.append(pillar["title"])
        parts.append(pillar["summary"])
        for section in pillar["sections"]:
            parts.append(section["section_title"])
            parts.append(section["summary"])
            for action in section["recommended_policy_actions"]:
                parts.append(action["action"])
    return join_parts(parts)


EXTRACTORS = {
    "PBIA.json": extract_pbia,
    "OCDE.json": extract_ocde,
    "AI_PLUS.json": extract_aiplus,
    "China_New_Gen.json": extract_china_new_gen,
    "Apply_AI_Strategy.json": extract_apply_ai_strategy,
    "AI_CONTINENT_ACTION_PLAN.json": extract_ai_continent,
    "AMERICA_AI_ACTION_PLAN.json": extract_america,
}

LABELS = {
    "PBIA.json": "PBIA (Brasil)",
    "AI_PLUS.json": "AI+ (China, 2023)",
    "China_New_Gen.json": "China New Gen (2017)",
    "Apply_AI_Strategy.json": "Apply AI Strategy (UE)",
    "AI_CONTINENT_ACTION_PLAN.json": "AI Continent Action Plan (UE)",
    "AMERICA_AI_ACTION_PLAN.json": "America's AI Action Plan (EUA)",
    "OCDE.json": "OECD Recommendation (referência do polo OCDE)",
}

# Mesma paleta categórica validada (skill dataviz) já usada em Estudo Gráfico.ipynb
# e Comparações.ipynb — mantém a identidade visual de cada documento consistente
# em todos os notebooks do projeto.
COLOR = {
    "PBIA.json": "#e34948",                      # red — identidade do PBIA
    "AMERICA_AI_ACTION_PLAN.json": "#008300",     # green
    "China_New_Gen.json": "#e87ba4",              # magenta
    "AI_CONTINENT_ACTION_PLAN.json": "#eda100",   # yellow
    "OCDE.json": "#898781",                       # cinza — âncora de referência, não polo posicionado
    "Apply_AI_Strategy.json": "#eb6834",          # orange
    "AI_PLUS.json": "#4a3aa7",                    # violet
}

SURFACE, INK_PRIMARY, INK_MUTED, GRIDLINE = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

raw_text = {}
for fn, extractor in EXTRACTORS.items():
    data = json.load(open(BASE / fn, encoding="utf-8"))
    raw_text[fn] = extractor(data)

doc_order = list(EXTRACTORS.keys())
for fn in doc_order:
    print(f"{LABELS[fn]:42s} {len(raw_text[fn]):7d} caracteres  ~{len(raw_text[fn].split()):5d} palavras (bruto)")

---
# 2. Tokenização e normalização morfológica (stemmer)

Stemmer idêntico ao de `Estudo Gráfico.ipynb` (dicionário curado de famílias
morfológicas + sufixos genéricos por remoção pura, nunca substituição
irregular) — sem alterações. Documentado e validado lá; reproduzido aqui na
íntegra para que este notebook seja executável de forma independente.

In [ ]:
MANUAL_STEM_OVERRIDES = {
    "development": "develop", "developing": "develop", "developed": "develop",
    "developments": "develop", "developers": "develop", "developer": "develop",
    "development-oriented": "develop",
    "technology": "technolog", "technologies": "technolog", "technological": "technolog",
    "technologically": "technolog", "technology-led": "technolog",
    "intelligence": "intelligen", "intelligent": "intelligen",
    "intelligentized": "intelligen", "intelligentization": "intelligen",
    "intelligentization-based": "intelligen", "intelligence-based": "intelligen",
    "intelligence-driven": "intelligen",
    "innovation": "innovat", "innovative": "innovat", "innovations": "innovat",
    "innovate": "innovat", "innovators": "innovat", "innovating": "innovat",
    "innovation-oriented": "innovat", "innovation-driven": "innovat",
    "innovation-style": "innovat", "innovation-friendly": "innovat",
    "industry": "industr", "industrial": "industr", "industries": "industr",
    "industry-specific": "industr", "industry-driven": "industr",
    "industry-education": "industr", "industry-leading": "industr",
    "governance": "govern", "government": "govern", "governmental": "govern",
    "governments": "govern", "governing": "govern", "government-procured": "govern",
    "security": "secur", "secure": "secur", "securely": "secur",
    "securing": "secur", "secured": "secur", "secure-by-design": "secur",
    "security-related": "secur",
    "sustainable": "sustain", "sustainability": "sustain", "sustained": "sustain",
    "responsible": "respons", "responsibility": "respons",
    "responsibilities": "respons", "responsibly": "respons",
    "strategic": "strateg", "strategy": "strateg", "strategies": "strateg",
    "transparency": "transpar", "transparent": "transpar", "transparently": "transpar",
    "cooperation": "cooperat", "cooperative": "cooperat", "cooperate": "cooperat",
    "investment": "invest", "investing": "invest", "investments": "invest",
    "creation": "creat", "creative": "creat", "creating": "creat",
    "creates": "creat", "created": "creat", "creativity": "creat",
    "creators": "creat", "create": "creat", "creational": "creat",
    "using": "use", "used": "use", "uses": "use",
    "ensuring": "ensur", "ensure": "ensur", "ensures": "ensur", "ensured": "ensur",
    "increasing": "increas", "increase": "increas", "increased": "increas",
    "increasingly": "increas", "increases": "increas", "incrementally": "increas",
    "promoting": "promot", "promote": "promot", "promotion": "promot",
    "promotes": "promot", "promoted": "promot", "promotable": "promot",
    "improving": "improv", "improve": "improv", "improvement": "improv",
    "improved": "improv", "improves": "improv",
    "strengthening": "strength", "strengthen": "strength",
    "strengthened": "strength", "strengthens": "strength",
    "generation": "generat", "generative": "generat", "generate": "generat",
    "generating": "generat", "generated": "generat", "generates": "generat",
    "generalization": "generat",
    "integration": "integrat", "integrated": "integrat",
    "integrating": "integrat", "integrate": "integrat", "integrative": "integrat",
    "regulation": "regulat", "regulatory": "regulat", "regulations": "regulat",
    "regulate": "regulat", "regulated": "regulat", "regulating": "regulat",
    "production": "produc", "products": "produc", "productivity": "produc",
    "product": "produc", "productive": "produc", "producers": "produc",
    "produced": "produc", "produce": "produc", "producer": "produc",
    "producing": "produc", "productions": "produc",
    "computing": "comput", "computational": "comput", "compute": "comput",
    "computer": "comput", "computerization": "comput", "computers": "comput",
    "computable": "comput",
    "advanced": "advanc", "advance": "advanc", "advances": "advanc",
    "advancing": "advanc", "advancement": "advanc", "advancements": "advanc",
    "digital": "digit", "digitalisation": "digit", "digitalized": "digit",
    "digital-intense": "digit", "digitally": "digit", "digitization": "digit",
    "discrimination": "discriminat", "discriminatory": "discriminat",
    "discriminate": "discriminat", "discriminated": "discriminat", "discriminates": "discriminat",
    "protect": "protect", "protects": "protect", "protected": "protect",
    "protecting": "protect", "protection": "protect", "protections": "protect",
    "protective": "protect",
    "bias": "bias", "biased": "bias", "biases": "bias",
    "diverse": "divers", "diversity": "divers", "diversify": "divers", "diversified": "divers",
    "equity": "equit", "equitable": "equit", "equitably": "equit",
    "accountability": "account", "accountable": "account", "accountably": "account",
    "inclusion": "inclus", "inclusive": "inclus", "inclusivity": "inclus", "inclusiveness": "inclus",
    "trustworthy": "trust", "trusted": "trust", "trusting": "trust", "trustworthiness": "trust",
    "fairness": "fair", "fairly": "fair",
    "safety": "safe", "safely": "safe", "unsafe": "unsafe",
    "vulnerable": "vulner", "vulnerability": "vulner", "vulnerabilities": "vulner",
    "consented": "consent", "consenting": "consent",
    "growth": "grow", "growing": "grow", "grows": "grow", "grew": "grow",
    "competitiveness": "competit", "competitive": "competit", "competition": "competit",
    "compete": "competit", "competing": "competit",
    "employment": "employ", "employed": "employ", "employer": "employ", "employers": "employ",
    "trading": "trade", "trader": "trade", "traders": "trade",
    "sovereignty": "sovereign",
    "defense": "defens", "defence": "defens", "defensive": "defens",
    "defend": "defens", "defended": "defens", "defending": "defens",
    "dominance": "domin", "dominant": "domin", "dominate": "domin", "dominated": "domin",
    "geopolitical": "geopolit", "geopolitics": "geopolit",
    "oversight": "oversee", "overseeing": "oversee",
    "institutional": "institution", "institutions": "institution",
    "compliance": "compl", "compliant": "compl", "comply": "compl", "complies": "compl",
    "semiconductors": "semiconductor",
    "supercomputers": "supercomputer", "supercomputing": "supercomputer",
    "skills": "skill", "skilled": "skill",
}

GENERIC_SUFFIXES = [
    ("ies", 3, "y"),
    ("ications", 4, ""), ("ication", 4, ""),
    ("izations", 4, ""), ("ization", 4, ""),
    ("ities", 4, ""), ("ity", 4, ""),
    ("ances", 4, ""), ("ance", 4, ""),
    ("ences", 4, ""), ("ence", 4, ""),
    ("ments", 4, ""), ("ment", 4, ""),
    ("nesses", 4, ""), ("ness", 4, ""),
    ("atively", 4, ""), ("ative", 4, ""),
    ("ically", 4, ""), ("ical", 4, ""),
    ("ives", 4, ""), ("ive", 4, ""),
    ("ally", 4, ""),
    ("ously", 4, ""), ("ous", 4, ""),
    ("ful", 4, ""),
    ("ably", 4, ""), ("ibly", 4, ""), ("able", 4, ""), ("ible", 4, ""),
    ("ancy", 4, ""), ("ency", 4, ""),
    ("als", 4, ""), ("al", 4, ""),
]
_PLURAL_S_EXCEPTIONS_SUFFIXES = ("ss", "us", "is", "ous")


def stem(word):
    # normaliza uma palavra já tokenizada (minúscula, alfabética) à sua forma aproximada de raiz
    if word in MANUAL_STEM_OVERRIDES:
        return MANUAL_STEM_OVERRIDES[word]
    for suffix, min_len, replacement in GENERIC_SUFFIXES:
        if word.endswith(suffix) and len(word) - len(suffix) >= min_len:
            return word[: -len(suffix)] + replacement
    if word.endswith("s") and not word.endswith(_PLURAL_S_EXCEPTIONS_SUFFIXES) and len(word) > 3:
        return word[:-1]
    return word


def tokenize(text, use_stem=True):
    toks = TOKEN_RE.findall(text.lower())
    toks = [t for t in toks if t not in STOPWORDS and len(t) > 2]
    if use_stem:
        toks = [stem(t) for t in toks]
    return toks


tokens, counts = {}, {}
for fn in doc_order:
    toks = tokenize(raw_text[fn])
    tokens[fn] = toks
    counts[fn] = Counter(toks)

for fn in doc_order:
    print(f"{LABELS[fn]:42s} {len(tokens[fn]):6d} tokens (pós-stem)  {len(counts[fn]):5d} termos únicos")

---
# 3. Os quatro polos — e a desproporção de tamanho entre eles

Cada polo é a soma dos documentos daquele bloco. UE e China combinam **dois**
documentos cada; EUA e OCDE são, cada um, **um único** documento — os quatro
polos **não têm o mesmo tamanho**, e essa desproporção precisa ser medida
antes de qualquer comparação, não descoberta depois.

In [ ]:
EU_DOCS = ["Apply_AI_Strategy.json", "AI_CONTINENT_ACTION_PLAN.json"]
CHINA_DOCS = ["AI_PLUS.json", "China_New_Gen.json"]
USA_DOCS = ["AMERICA_AI_ACTION_PLAN.json"]
OECD_DOCS = ["OCDE.json"]


def pool(doclist):
    c, t = Counter(), 0
    for fn in doclist:
        c += counts[fn]
        t += len(tokens[fn])
    return c, t


POLES = {
    "UE": pool(EU_DOCS),
    "China": pool(CHINA_DOCS),
    "EUA": pool(USA_DOCS),
    "OCDE": pool(OECD_DOCS),
}

pole_sizes = {p: t for p, (c, t) in POLES.items()}
maior, menor = max(pole_sizes.values()), min(pole_sizes.values())
print("Tamanho de cada polo (tokens pós-stem):")
for p, t in pole_sizes.items():
    print(f"  {p:6s} {t:6d} tokens   ({t / maior * 100:5.1f}% do maior polo)")
print(f"\nRazão maior/menor polo: {maior / menor:.2f}x  (UE tem {pole_sizes['UE'] / pole_sizes['OCDE']:.1f}x mais tokens que a OCDE)")
print("\n=> Toda comparação de vocabulário a partir daqui usa frequência relativa (por 1.000 tokens),")
print("   nunca contagem bruta, exatamente para neutralizar esta desproporção.")

---
# 4. Vocabulário distintivo de cada polo — teste de *keyness* (G²)

**Método**: log-likelihood ratio (G², Dunning 1993) — o mesmo teste já usado
em `Estudo Gráfico.ipynb` (Seções 5 e 7.2) — comparando, para cada polo, sua
frequência relativa de cada termo contra a frequência relativa do **mesmo
termo somado nos outros três polos**. G² > 10,83 corresponde a p < 0,001.
**Piso mínimo de evidência**: frequência combinada (polo + resto) ≥ 5, para
que termos raríssimos (instáveis estatisticamente) não dominem a lista.

**Duas exclusões, ambas com evidência auditável abaixo:**

1. **Termos autorreferentes** (`SELF_REFERENTIAL`) — nomes do próprio
   bloco/país/instituição (ex.: "china", "european", "america") são triviais:
   é óbvio que o polo China menciona "china" mais que os outros. Mesma
   exclusão de `Estudo Gráfico.ipynb`.
2. **Ruído estrutural** (`STRUCTURAL_NOISE`) — rótulos de estrutura do
   documento (nome de seção/categoria), não vocabulário temático. Esta
   exclusão é **nova neste notebook** e nasceu de uma auditoria: o termo
   "overview" aparecia como uma das palavras mais distintivas do polo UE
   (G²=32,4) — mas a célula abaixo mostra que, das 19 ocorrências no polo UE,
   **nenhuma está no corpo do texto**: todas vêm do rótulo de subtítulo
   genérico "Overview"/"Sector overview"/"Defence overview" que antecede
   cada subseção setorial da Apply AI Strategy e do AI Continent Action
   Plan. Da mesma forma, "preamble" (11/11 ocorrências) e "definition"
   (5/5) no polo OCDE vêm inteiramente do rótulo de categoria da seção
   ("Preamble", "Definitions"), repetido uma vez por cláusula — não de
   conteúdo substantivo. "section" tem a mesma origem (rótulos "Section 1:
   ...", "Section 2: ..."). Note que termos vizinhos e igualmente frequentes
   — "single" (mercado único) e "uptake" (adoção de IA) — foram checados da
   mesma forma e **mantidos**, porque aparecem quase inteiramente no corpo
   do texto (ver saída abaixo).

In [ ]:
SELF_REFERENTIAL = {
    "pbia", "brazil", "brazilian", "sus", "finep",
    "china", "chinese", "european", "europe", "commission", "eu", "union",
    "america", "american", "united", "states", "trump", "brics", "ocde", "oecd",
    "washington", "beijing", "brussels",
}

# --- auditoria que motivou STRUCTURAL_NOISE: onde cada termo aparece de fato? ---
def audit_term(term, fn_list, label):
    body_hits, structural_hits = 0, 0
    for fn in fn_list:
        data = json.load(open(BASE / fn, encoding="utf-8"))
        for s in data.get("sections", []):
            if term in s.get("text", "").lower():
                body_hits += s["text"].lower().count(term)
            for field in ("category", "subtitle"):
                if term in (s.get(field) or "").lower():
                    structural_hits += 1
    print(f"  '{term}' em {label:26s} corpo={body_hits:3d}  rótulo(categoria/subtítulo)={structural_hits:3d}")


print("Auditoria: origem das ocorrências (corpo do texto vs. rótulo estrutural)")
audit_term("overview", EU_DOCS, "polo UE")
audit_term("single", EU_DOCS, "polo UE (controle)")
audit_term("uptake", EU_DOCS, "polo UE (controle)")
audit_term("preamble", OECD_DOCS, "polo OCDE")
audit_term("definition", OECD_DOCS, "polo OCDE")
audit_term("section", OECD_DOCS, "polo OCDE")

STRUCTURAL_NOISE = {"overview", "preamble", "section", "definition"}
EXCLUDE = SELF_REFERENTIAL | STRUCTURAL_NOISE
print(f"\n{len(STRUCTURAL_NOISE)} termos excluídos por ruído estrutural: {sorted(STRUCTURAL_NOISE)}")
print(f"{len(SELF_REFERENTIAL)} termos excluídos por autorreferência.")

In [ ]:
def log_likelihood(a, c_, b, d):
    # a = freq do termo no corpus A; c_ = total de tokens do corpus A
    # b = freq do termo no corpus B; d = total de tokens do corpus B
    # retorna G² (razão de verossimilhança, ~qui-quadrado, 1 g.l.)
    if a == 0 and b == 0:
        return 0.0
    e1 = c_ * (a + b) / (c_ + d)
    e2 = d * (a + b) / (c_ + d)
    g2 = 0.0
    if a > 0:
        g2 += a * math.log(a / e1)
    if b > 0:
        g2 += b * math.log(b / e2)
    return 2 * g2


MIN_COUNT_KEYNESS = 5


def distinctive_terms(pole_name):
    gc, gt = POLES[pole_name]
    rc, rt = Counter(), 0
    for other in POLES:
        if other == pole_name:
            continue
        oc, ot = POLES[other]
        rc += oc
        rt += ot
    rows = []
    for term in set(gc) | set(rc):
        if term in EXCLUDE:
            continue
        a, b = gc.get(term, 0), rc.get(term, 0)
        if a + b < MIN_COUNT_KEYNESS:
            continue
        g2 = log_likelihood(a, gt, b, rt)
        rate_g, rate_r = a / gt * 1000, b / rt * 1000
        if rate_g <= rate_r:  # só termos que puxam PARA o polo, não para o resto
            continue
        rows.append((term, a, b, g2, rate_g, rate_r))
    rows.sort(key=lambda r: (-r[3], r[0]))  # desempate alfabético -> reprodutível entre execuções
    return [r for r in rows if r[3] > 10.83]


sig_terms = {pole: distinctive_terms(pole) for pole in POLES}
for pole, rows in sig_terms.items():
    print(f"{pole:6s}: {len(rows)} termos estatisticamente distintivos (G²>10,83, p<0,001)")

---
# 5. Vocabulário fixo por polo — por que top-15 e não "todos os significativos"

Os quatro polos têm **números muito diferentes de termos significativos**
(Seção 4: 108 na UE e 107 na China, contra 32 nos EUA e 36 na OCDE) — reflexo
direto da Seção 3: polos maiores (UE, China somam dois documentos) dão mais
poder estatístico ao teste G², que captura mais termos. Se cada eixo usasse
"todos os termos significativos" do seu polo, o eixo X (UE-China) somaria
sobre ~3x mais termos que o eixo Y (EUA-OCDE) — inflando artificialmente a
amplitude do eixo X por causa do tamanho da amostra, não por diferença real
de vocabulário.

**Correção**: cada polo contribui um número **fixo** de termos para o
escore — os **top 15 por G²**. Isso equaliza os quatro polos no mesmo
critério (os 15 termos estatisticamente mais fortes de cada um), em vez de
deixar o tamanho da amostra determinar quantos termos cada lado do gráfico
usa. A Seção 8 testa a sensibilidade desta escolha (N=10, 15, 20, 25) e
mostra que o posicionamento por quadrante é estável para 5 dos 6 documentos.

Note a assimetria que **permanece mesmo após esta correção**, e que é
importante para a leitura do gráfico: os top-15 da UE/China vêm de um total
de 107-108 candidatos (top-15 é ~14% do vocabulário distintivo disponível
daquele polo), enquanto os top-15 dos EUA/OCDE vêm de apenas 32-36
candidatos (top-15 é ~42-47% do vocabulário distintivo disponível). O eixo
Y usa uma fatia proporcionalmente **mais completa** do vocabulário
distintivo dos seus dois polos do que o eixo X usa do seu — consequência
direta de EUA e OCDE serem, cada um, um único documento menor, não um
viés introduzido pelo método.

In [ ]:
N_AXIS = 15
VOCAB_SET = {pole: {r[0] for r in rows[:N_AXIS]} for pole, rows in sig_terms.items()}

# checagem de independência dos eixos: nenhum termo pode aparecer em mais de um polo
overlaps = [(p1, p2, VOCAB_SET[p1] & VOCAB_SET[p2])
            for i, p1 in enumerate(POLES) for p2 in list(POLES)[i + 1:]]
overlaps = [o for o in overlaps if o[2]]
print(f"Sobreposição entre os vocabulários dos 4 polos: {overlaps if overlaps else 'nenhuma (eixos independentes)'}")
print()

STEM_DISPLAY_LABEL = {
    "digit": "digital", "strateg": "strategy/-ic", "appl": "application",
    "intelligen": "intelligent/-ce", "technolog": "technology",
    "promot": "promote/-ion", "strength": "strengthen", "develop": "development",
    "feder": "federal", "vulner": "vulnerable/-ability", "crit": "critical",
    "secur": "security/-e", "respons": "responsible/-ility", "internation": "international",
}

for pole, rows in sig_terms.items():
    print(f"--- {pole}: os {N_AXIS} termos usados no escore deste polo ---")
    for r in rows[:N_AXIS]:
        term, a, b, g2, rate_g, rate_r = r
        disp = STEM_DISPLAY_LABEL.get(term, term)
        print(f"  {disp:20s} G²={g2:7.1f}  no polo={a:4d} ({rate_g:6.2f}‰)  no resto={b:4d} ({rate_r:5.2f}‰)")
    print()

---
# 6. Vocabulário comum entre os quatro blocos

Termos que aparecem nos **quatro** polos (frequência ≥ 2 em cada um) e que
**não** foram selecionados como distintivos de nenhum deles — o registro
partilhado de política de IA que atravessa UE, China, EUA e OCDE, e que
portanto **não** carrega sinal de eixo nenhum. É importante nomeá-lo
explicitamente: mostra que os quatro documentos-polo não são mundos
totalmente separados — dividem uma base comum — e serve de contraponto ao
vocabulário exclusivo da Seção 5.

In [ ]:
all_pole_terms = set()
for c, t in POLES.values():
    all_pole_terms |= set(c)
distinctive_any = set()
for rows in sig_terms.values():
    distinctive_any |= {r[0] for r in rows}

common_rows = []
for term in all_pole_terms:
    if term in EXCLUDE or term in distinctive_any:
        continue
    by_pole = {p: POLES[p][0].get(term, 0) for p in POLES}
    if min(by_pole.values()) < 2:
        continue
    common_rows.append((term, sum(by_pole.values()), by_pole))
common_rows.sort(key=lambda r: (-r[1], r[0]))

print(f"{len(common_rows)} termos comuns aos 4 polos (freq >= 2 em cada, não distintivo de nenhum)\n")
print(f"{'termo':14s} {'total':>6s} {'UE':>5s} {'China':>6s} {'EUA':>5s} {'OCDE':>5s}")
for r in common_rows[:20]:
    term, total, by_pole = r
    disp = STEM_DISPLAY_LABEL.get(term, term)
    print(f"{disp:14s} {total:6d} {by_pole['UE']:5d} {by_pole['China']:6d} {by_pole['EUA']:5d} {by_pole['OCDE']:5d}")

---
# 7. Cálculo das coordenadas — densidade relativa de cada vocabulário-polo

Para cada documento a posicionar, calculamos a **densidade** de cada
vocabulário-polo nesse documento: soma das ocorrências dos 15 termos do polo,
dividida pelo total de tokens do próprio documento, em partes por mil (‰) —
a mesma frequência relativa usada em todo o projeto, o que garante que o
tamanho do documento sendo posicionado (PBIA tem 7.562 tokens; AI+ tem
apenas 2.197) não influencie o escore.

$$X = \text{densidade}_{UE} - \text{densidade}_{China} \qquad\qquad Y = \text{densidade}_{EUA} - \text{densidade}_{OCDE}$$

**Nota metodológica importante**: os documentos que definem um polo tendem,
por construção, a pontuar alto nesse mesmo polo (é o vocabulário mais
característico deles, por definição). Isso não é um viés a esconder — é
exatamente o que valida o método (se a Apply AI Strategy não pontuasse alto
em X positivo, o vocabulário-polo estaria mal escolhido). O que é
genuinamente informativo, e não garantido por construção, é: **(a)** a
posição relativa *dentro* do mesmo bloco (AI Continent vs. Apply AI
Strategy; China New Gen vs. AI+); e **(b)** onde caem os documentos que
**não** definem nenhum polo — America's AI Action Plan no eixo X, e,
sobretudo, o **PBIA nos dois eixos**.

In [ ]:
PLOT_DOCS = ["AI_CONTINENT_ACTION_PLAN.json", "Apply_AI_Strategy.json", "China_New_Gen.json",
             "AI_PLUS.json", "AMERICA_AI_ACTION_PLAN.json", "PBIA.json", "OCDE.json"]


def density(vocab_set, fn):
    c = counts[fn]
    total = len(tokens[fn])
    hits = sum(c.get(t, 0) for t in vocab_set)
    return hits / total * 1000


coords = {}
print(f"{'documento':32s} {'UE‰':>8s} {'China‰':>8s} {'EUA‰':>8s} {'OCDE‰':>8s} {'X':>9s} {'Y':>9s}")
for fn in PLOT_DOCS:
    eu_d = density(VOCAB_SET["UE"], fn)
    ch_d = density(VOCAB_SET["China"], fn)
    us_d = density(VOCAB_SET["EUA"], fn)
    oe_d = density(VOCAB_SET["OCDE"], fn)
    x, y = eu_d - ch_d, us_d - oe_d
    coords[fn] = (x, y)
    print(f"{LABELS[fn]:32s} {eu_d:8.2f} {ch_d:8.2f} {us_d:8.2f} {oe_d:8.2f} {x:9.2f} {y:9.2f}")

---
# 8. Checagem de robustez — o posicionamento muda com N?

Recalculamos X e Y para N = 10, 15, 20 e 25 termos por polo. Se o
**quadrante** de um documento muda dependendo dessa escolha, qualquer leitura
sobre ele precisa ser tratada como frágil; se não muda, a leitura é robusta
à escolha de N — o mesmo espírito da checagem de robustez da Seção 4 de
`Estudo Gráfico.ipynb`.

In [ ]:
print(f"{'documento':32s}" + "".join(f"   N={n:<2d} (X,Y)".rjust(20) for n in [10, 15, 20, 25]))
robust_coords = {n: {} for n in [10, 15, 20, 25]}
for n in [10, 15, 20, 25]:
    vocab_n = {pole: {r[0] for r in rows[:n]} for pole, rows in sig_terms.items()}
    for fn in PLOT_DOCS:
        eu_d = density(vocab_n["UE"], fn)
        ch_d = density(vocab_n["China"], fn)
        us_d = density(vocab_n["EUA"], fn)
        oe_d = density(vocab_n["OCDE"], fn)
        robust_coords[n][fn] = (eu_d - ch_d, us_d - oe_d)

for fn in PLOT_DOCS:
    row = f"{LABELS[fn]:32s}"
    for n in [10, 15, 20, 25]:
        x, y = robust_coords[n][fn]
        row += f"  ({x:6.1f},{y:6.1f})"
    print(row)

**Leitura da robustez**: para os cinco documentos que definem um polo
(AI Continent, Apply AI Strategy, China New Gen, AI+, America's AI Action
Plan), o **sinal** de X e de Y é idêntico nas quatro colunas — a magnitude
varia, mas o quadrante nunca muda. Para a **OCDE** (âncora, não documento
posicionado), Y é sempre fortemente negativo, como esperado por construção.

Para o **PBIA**, X é sempre negativo e de magnitude pequena (-13 a -38, ante
-107 a -205 dos documentos chineses de fato) em todas as colunas — um sinal
estável. **Y, porém, muda de sinal** entre N=15 (+6,5) e N=20 (-9,5), sempre
com magnitude pequena (nunca acima de ~9,5, ante +46 a +87 do America's AI
Action Plan e -104 a -211 da OCDE). Tratamos isso como o próprio resultado,
não como uma falha do método: **o PBIA não tem uma inclinação robusta na
direção de nenhum dos dois polos do eixo Y** — ele ocupa a vizinhança da
origem, e o sinal exato depende de detalhes de corte que não deveriam ser
lidos como uma conclusão forte.

---
# 9. A bússola — gráfico de posicionamento

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

ax.axhline(0, color=GRIDLINE, linewidth=1.3, zorder=1)
ax.axvline(0, color=GRIDLINE, linewidth=1.3, zorder=1)

MAIN_DOCS = [fn for fn in PLOT_DOCS if fn != "OCDE.json"]

LABEL_OFFSET = {
    "PBIA.json": (10, -14),
    "AI_CONTINENT_ACTION_PLAN.json": (10, 8),
    "Apply_AI_Strategy.json": (10, -16),
    "China_New_Gen.json": (-14, 10),
    "AI_PLUS.json": (-14, -16),
    "AMERICA_AI_ACTION_PLAN.json": (10, 8),
}

for fn in MAIN_DOCS:
    x, y = coords[fn]
    is_pbia = fn == "PBIA.json"
    ax.scatter([x], [y], s=380 if is_pbia else 240, color=COLOR[fn], zorder=4,
               edgecolor=SURFACE, linewidth=1.6,
               marker="*" if is_pbia else "o")
    dx, dy = LABEL_OFFSET[fn]
    ax.annotate(LABELS[fn], (x, y), xytext=(dx, dy), textcoords="offset points",
                fontsize=10.5 if is_pbia else 9.5,
                fontweight="bold" if is_pbia else "normal",
                color=INK_PRIMARY, ha="left" if dx >= 0 else "right", va="center")

# OCDE: âncora de referência do polo Y negativo, não um documento "posicionado" em
# pé de igualdade -- marcador vazado, cor recessiva, para não competir com os 6 pontos
# categóricos acima
ox, oy = coords["OCDE.json"]
ax.scatter([ox], [oy], s=200, facecolor="none", edgecolor=COLOR["OCDE.json"],
           linewidth=1.8, zorder=3, marker="D")
ax.annotate("OCDE (âncora do polo Y-, não é\num documento posicionado)", (ox, oy),
            xytext=(10, -14), textcoords="offset points", fontsize=8.5,
            color=INK_MUTED, ha="left", va="center", style="italic")

xs = [coords[fn][0] for fn in PLOT_DOCS]
ys = [coords[fn][1] for fn in PLOT_DOCS]
xpad = (max(xs) - min(xs)) * 0.22
ypad = (max(ys) - min(ys)) * 0.16
ax.set_xlim(min(xs) - xpad, max(xs) + xpad)
ax.set_ylim(min(ys) - ypad, max(ys) + ypad)

xlo, xhi = ax.get_xlim()
ylo, yhi = ax.get_ylim()

eu_terms = ", ".join(STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in sig_terms["UE"][:4])
china_terms = ", ".join(STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in sig_terms["China"][:4])
usa_terms = ", ".join(STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in sig_terms["EUA"][:6])
oecd_terms = ", ".join(STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in sig_terms["OCDE"][:6])

# Rótulos de polo do eixo X deslocados para longe de y=0 (onde China, UE e o
# próprio PBIA estão a menos de 10 unidades de distância) e escalonados em duas
# alturas diferentes -- mesmo com só 4 termos cada, um texto ancorado em xlo
# estendendo-se para a direita e outro ancorado em xhi estendendo-se para a
# esquerda podem se encontrar no meio; alturas diferentes evitam a colisão.
ax.text(xhi, ylo * 0.30, f"UE →\n{eu_terms}", ha="right", va="center", fontsize=8.5,
        color=COLOR["AI_CONTINENT_ACTION_PLAN.json"], fontweight="bold")
ax.text(xlo, ylo * 0.45, f"← China\n{china_terms}", ha="left", va="center", fontsize=8.5,
        color=COLOR["China_New_Gen.json"], fontweight="bold")
ax.text(0, yhi, f"↑ EUA\n{usa_terms}", ha="left", va="top", fontsize=8.5,
        color=COLOR["AMERICA_AI_ACTION_PLAN.json"], fontweight="bold")
ax.text(0, ylo, f"↓ OCDE\n{oecd_terms}", ha="left", va="bottom", fontsize=8.5,
        color=COLOR["OCDE.json"], fontweight="bold")

ax.set_xlabel(f"Eixo X — densidade relativa (‰): vocabulário UE menos vocabulário China  (N={N_AXIS} termos/polo)",
              color=INK_MUTED, fontsize=9.5)
ax.set_ylabel(f"Eixo Y — densidade relativa (‰): vocabulário EUA menos vocabulário OCDE  (N={N_AXIS} termos/polo)",
              color=INK_MUTED, fontsize=9.5)
ax.set_title("A bússola de vocabulário: PBIA entre UE, China, EUA e OCDE",
             color=INK_PRIMARY, fontsize=14.5, fontweight="bold", pad=16)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED)
ax.grid(True, color=GRIDLINE, linewidth=0.6, zorder=0, alpha=0.6)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

### 9.1. Versão para publicação — eixos padronizados, rótulos temáticos

A versão abaixo usa **exatamente as mesmas coordenadas** calculadas na
Seção 7 (nada é recalculado) — só a apresentação muda, em quatro pontos:

1. **Eixos padronizados em ±200‰** e proporção 1:1 entre X e Y (em vez dos
   limites ajustados aos dados da versão técnica acima) — torna a
   *magnitude* comparável a olho nu entre os dois eixos, e deixa claro,
   pela distância à borda, o quanto cada ponto está longe do polo que ele
   mais representa.
2. Nos quatro rótulos de polo, a **lista de termos crus** (Seção 5) é
   substituída pela **síntese temática já justificada, termo a termo, na
   Seção 11** — ex.: o polo EUA aparece como "competição geopolítica e
   administração federal" em vez de "federal, agency, adversary...". A
   lista de termos originais continua disponível nas Seções 5 e 10 para
   quem quiser a evidência lexical bruta por trás de cada síntese.
3. **PBIA em losango** e **OCDE em quadrado azul** — reservando o formato
   de estrela para nenhum documento e deixando os dois pontos de leitura
   mais delicada (o centro da análise e a âncora de referência) com
   identidade visual própria, além da cor.
4. Uma **legenda formal** (cor + forma) substitui parte da rotulagem
   direta, reduzindo o texto solto sobre a área do gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

AXIS_LIMIT = 200
ax.axhline(0, color=GRIDLINE, linewidth=1.4, zorder=1)
ax.axvline(0, color=GRIDLINE, linewidth=1.4, zorder=1)
ax.set_xlim(-AXIS_LIMIT, AXIS_LIMIT)
ax.set_ylim(-AXIS_LIMIT, AXIS_LIMIT)
ax.set_aspect("equal", adjustable="box")

# cor da OCDE sobrescrita apenas nesta figura (azul, #2a78d6 -- mesmo azul já
# validado e usado para identidade categórica em Comparações.ipynb/Estudo
# Gráfico.ipynb) -- o dicionário COLOR global, usado pelas demais células
# deste notebook, não é alterado.
COLOR_PUB = dict(COLOR)
COLOR_PUB["OCDE.json"] = "#2a78d6"

# nomes de exibição sobrescritos só nesta figura -- o dicionário LABELS global,
# usado no restante do notebook (inclusive nas Seções 5, 10 e 11, onde o ano de
# cada plano chinês é analiticamente relevante), não é alterado.
LABELS_PUB = dict(LABELS)
LABELS_PUB["China_New_Gen.json"] = "New Generation AI (China)"
LABELS_PUB["AI_PLUS.json"] = "AI+ (China)"

MARKER_PUB = {fn: "o" for fn in PLOT_DOCS}
MARKER_PUB["PBIA.json"] = "D"
MARKER_PUB["OCDE.json"] = "s"

SIZE_PUB = {fn: 620 for fn in PLOT_DOCS}
SIZE_PUB["PBIA.json"] = 780
SIZE_PUB["OCDE.json"] = 680

LABEL_OFFSET_PUB = {
    "PBIA.json": (0, -24),
    "AI_CONTINENT_ACTION_PLAN.json": (16, 12),
    "Apply_AI_Strategy.json": (16, -18),
    # China New Gen e AI+ ficam a poucas unidades da borda esquerda (x=-159 e
    # x=-149, contra um limite fixo de -200) -- um rótulo estendendo-se para a
    # esquerda (dx<0) ultrapassa a borda do eixo e invade a área do rótulo do
    # eixo Y. Por isso os dois usam só deslocamento vertical (dx=0, centralizado
    # no ponto), um acima e outro abaixo, já que os dois pontos estão a menos
    # de 10 unidades um do outro em y.
    "China_New_Gen.json": (0, 28),
    "AI_PLUS.json": (0, -28),
    "AMERICA_AI_ACTION_PLAN.json": (16, 10),
    "OCDE.json": (18, -8),
}

for fn in PLOT_DOCS:
    x, y = coords[fn]
    is_pbia = fn == "PBIA.json"
    # todos os marcadores usam um contorno na cor do fundo (SURFACE), só para
    # separar visualmente o ponto da grade por trás -- o PBIA é a exceção
    # pedida: contorno preto fino de verdade, para destacá-lo como o centro
    # da análise.
    edge_color = INK_PRIMARY if is_pbia else SURFACE
    edge_width = 2 if is_pbia else 2.0
    ax.scatter([x], [y], s=SIZE_PUB[fn], color=COLOR_PUB[fn], marker=MARKER_PUB[fn],
               zorder=4, edgecolor=edge_color, linewidth=edge_width)
    dx, dy = LABEL_OFFSET_PUB[fn]
    ax.annotate(LABELS_PUB[fn], (x, y), xytext=(dx, dy), textcoords="offset points",
                fontsize=14 if is_pbia else 12, fontweight="bold" if is_pbia else "normal",
                color=INK_PRIMARY, ha="center" if dx == 0 else ("left" if dx > 0 else "right"),
                va="top" if dy < 0 else "bottom")

# --- síntese temática de cada polo (justificada termo a termo na Seção 11),
# no lugar da lista de termos crus -- mais legível para quem não acompanhou
# o pipeline estatístico das Seções 4-5.
POLE_THEME = {
    "UE": "Arquitetura regulatório-institucional\nmultinacional",
    "China": "Arquitetura técnica de\nsistemas de IA",
    "EUA": "Competição geopolítica e\nadministração federal",
    "OCDE": "Instrumento jurídico multilateral de\ndireitos e governança (soft law)",
}
POLE_COLOR = {"UE": COLOR["AI_CONTINENT_ACTION_PLAN.json"], "China": COLOR["China_New_Gen.json"],
              "EUA": COLOR["AMERICA_AI_ACTION_PLAN.json"], "OCDE": COLOR_PUB["OCDE.json"]}

# China e UE na MESMA linha (y=-60, logo abaixo da grade de y=-50) -- os textos
# são curtos o bastante (síntese temática, não mais a lista de termos crus) para
# não se encontrarem no meio do gráfico mesmo compartilhando a altura.
POLE_LABEL_Y_XAXIS = -60
ax.text(AXIS_LIMIT * 0.97, POLE_LABEL_Y_XAXIS, f"UE →\n{POLE_THEME['UE']}", ha="right", va="center",
        fontsize=11, color=POLE_COLOR["UE"], fontweight="bold")
ax.text(-AXIS_LIMIT * 0.97, POLE_LABEL_Y_XAXIS, f"← China\n{POLE_THEME['China']}", ha="left", va="center",
        fontsize=11, color=POLE_COLOR["China"], fontweight="bold")

# EUA e OCDE deslocados um pouco para a direita do zero (x=15, não x=0) para não
# ficarem colados na linha do eixo Y -- e alinhados entre si no mesmo x.
POLE_LABEL_X_YAXIS = 15
ax.text(POLE_LABEL_X_YAXIS, AXIS_LIMIT * 0.97, f"↑ EUA\n{POLE_THEME['EUA']}", ha="left", va="top",
        fontsize=11, color=POLE_COLOR["EUA"], fontweight="bold")
ax.text(POLE_LABEL_X_YAXIS, -AXIS_LIMIT * 0.97, f"↓ OCDE\n{POLE_THEME['OCDE']}", ha="left", va="bottom",
        fontsize=11, color=POLE_COLOR["OCDE"], fontweight="bold")

# --- legenda formal (cor + forma), no canto vazio superior-esquerdo ---
legend_handles = [
    plt.Line2D([0], [0], marker=MARKER_PUB[fn], color="none", markerfacecolor=COLOR_PUB[fn],
               markeredgecolor=(INK_PRIMARY if fn == "PBIA.json" else SURFACE), markersize=13, label=LABELS_PUB[fn])
    for fn in PLOT_DOCS
]
legend = ax.legend(handles=legend_handles, loc="upper left", frameon=True, fontsize=10,
                    labelcolor=INK_PRIMARY, borderpad=1.0, labelspacing=1.05,
                    title="Documentos", title_fontsize=11)
legend.get_frame().set_facecolor(SURFACE)
legend.get_frame().set_edgecolor(GRIDLINE)
legend.get_title().set_color(INK_PRIMARY)

ax.set_xlabel("EIXO X — Vocabulário específico do bloco China vs. bloco UE",
              color=INK_MUTED, fontsize=11)
ax.set_ylabel("EIXO Y — Vocabulário específico do polo OCDE vs. bloco EUA",
              color=INK_MUTED, fontsize=11)

ax.set_title("Posicionamento relativo do PBIA frente aos blocos UE, China, EUA e OCDE",
             color=INK_MUTED, fontsize=10.5, pad=12)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=10)
ax.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0, alpha=0.6)
ax.set_axisbelow(True)

# Três níveis de título, do mais genérico ao mais técnico -- fig.suptitle (título
# principal), fig.text (linha dos quatro atores) e ax.set_title (subtítulo
# metodológico, já definido acima) -- todos ACIMA da área reservada pelo
# tight_layout(rect=...), o que evita a sobreposição entre eles.
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.suptitle("Posicionamento do Plano Brasileiro de IA (PBIA) no Cenário Internacional",
             color=INK_PRIMARY, fontsize=17, fontweight="bold", y=0.985)
fig.text(0.5, 0.94, "PBIA (Brasil), China, União Europeia e Estados Unidos",
         ha="center", va="top", color=INK_PRIMARY, fontsize=14)
plt.show()

---
# 10. Quanto do vocabulário de cada polo o PBIA realmente usa

Para cada um dos quatro polos, os 8 termos mais fortes (G²) do polo,
comparando a frequência relativa no próprio polo com a frequência relativa
no PBIA — a evidência direta por trás da posição do PBIA no gráfico da
Seção 9.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11), facecolor=SURFACE)

pole_color = {"UE": COLOR["AI_CONTINENT_ACTION_PLAN.json"], "China": COLOR["China_New_Gen.json"],
              "EUA": COLOR["AMERICA_AI_ACTION_PLAN.json"], "OCDE": COLOR["OCDE.json"]}
pole_title = {"UE": "Polo UE\n(Apply AI Strategy + AI Continent)", "China": "Polo China\n(AI+ + China New Gen)",
              "EUA": "Polo EUA\n(America's AI Action Plan)", "OCDE": "Polo OCDE\n(referência do eixo Y-)"}

pbia_total = len(tokens["PBIA.json"])
pbia_counts = counts["PBIA.json"]

for ax, pole in zip(axes.flat, ["UE", "China", "EUA", "OCDE"]):
    ax.set_facecolor(SURFACE)
    rows = sig_terms[pole][:8][::-1]
    labels = [STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in rows]
    pole_rates = [r[4] for r in rows]
    pbia_rates = [pbia_counts.get(r[0], 0) / pbia_total * 1000 for r in rows]
    y_pos = np.arange(len(rows))
    bar_h = 0.35
    ax.barh(y_pos + bar_h / 2, pole_rates, height=bar_h, color=pole_color[pole], zorder=3, label="no polo")
    ax.barh(y_pos - bar_h / 2, pbia_rates, height=bar_h, color=COLOR["PBIA.json"], zorder=3, label="no PBIA")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=10)
    ax.set_xlabel("Frequência relativa (‰)", color=INK_MUTED, fontsize=9)
    ax.set_title(pole_title[pole], color=INK_PRIMARY, fontsize=11.5, fontweight="bold", pad=10)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.suptitle("Os termos mais fortes de cada polo (G²) vs. quanto o PBIA usa cada um",
             color=INK_PRIMARY, fontsize=14, fontweight="bold", y=1.0)
fig.tight_layout()
plt.show()

## 10.1. Os 15 termos mais próprios do PBIA — e quanto cada polo os usa

O gráfico anterior parte do vocabulário de cada **polo** e mede quanto o
PBIA usa dele. Esta seção inverte a pergunta: quais são os termos mais
**distintivos do próprio PBIA** — mesmo teste de *keyness* (G², Seção 4),
agora com o **PBIA no papel de polo** e os **quatro polos combinados**
como "resto" — e, para esses 15 termos, quanto cada um dos quatro polos
efetivamente os usa. É o vocabulário que o PBIA **não herda** de nenhum
dos quatro blocos: próprio o bastante para ser estatisticamente
distintivo do documento brasileiro quando comparado aos quatro juntos.


In [ ]:
pbia_rest_counts, pbia_rest_total = Counter(), 0
for c, t in POLES.values():
    pbia_rest_counts += c
    pbia_rest_total += t

pbia_rows = []
for term in set(pbia_counts) | set(pbia_rest_counts):
    if term in EXCLUDE:
        continue
    a, b = pbia_counts.get(term, 0), pbia_rest_counts.get(term, 0)
    if a + b < MIN_COUNT_KEYNESS:
        continue
    g2 = log_likelihood(a, pbia_total, b, pbia_rest_total)
    rate_pbia, rate_rest = a / pbia_total * 1000, b / pbia_rest_total * 1000
    if rate_pbia <= rate_rest:  # só termos que puxam PARA o PBIA, não para os polos
        continue
    pbia_rows.append((term, a, b, g2, rate_pbia, rate_rest))
pbia_rows.sort(key=lambda r: (-r[3], r[0]))  # mesmo desempate alfabético da Seção 4
pbia_distinctive = [r for r in pbia_rows if r[3] > 10.83]

N_PBIA = 15
top_pbia_terms = pbia_distinctive[:N_PBIA]

print(f"{len(pbia_distinctive)} termos estatisticamente distintivos do PBIA (G²>10,83, p<0,001) vs. os 4 polos combinados\n")
print(f"{'termo':20s} {'G²':>7s} {'PBIA‰':>8s} {'polos‰':>8s}")
for r in top_pbia_terms:
    term, a, b, g2, rate_p, rate_r = r
    disp = STEM_DISPLAY_LABEL.get(term, term)
    print(f"{disp:20s} {g2:7.1f} {rate_p:8.2f} {rate_r:8.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

rows = top_pbia_terms[::-1]  # menor G² embaixo, maior no topo -- convenção da Seção 10
labels = [STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in rows]
pbia_rates = [r[4] for r in rows]
y_pos = np.arange(len(rows))

ax.barh(y_pos, pbia_rates, height=0.55, color=COLOR["PBIA.json"], zorder=3,
        label=LABELS["PBIA.json"])

POLE_MARKER = {"UE": "o", "China": "s", "EUA": "^", "OCDE": "D"}
pole_color = {"UE": COLOR["AI_CONTINENT_ACTION_PLAN.json"], "China": COLOR["China_New_Gen.json"],
              "EUA": COLOR["AMERICA_AI_ACTION_PLAN.json"], "OCDE": COLOR["OCDE.json"]}

for pole, marker in POLE_MARKER.items():
    pc, pt = POLES[pole]
    rates = [pc.get(r[0], 0) / pt * 1000 for r in rows]
    ax.scatter(rates, y_pos, s=70, color=pole_color[pole], marker=marker, zorder=4,
               edgecolor=SURFACE, linewidth=1.2, label=pole)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Frequência relativa (‰)", color=INK_MUTED, fontsize=9.5)
ax.set_title(f"Os {N_PBIA} termos mais próprios do PBIA (G²) vs. quanto cada polo os usa",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", pad=12)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(colors=INK_MUTED)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 10.2. Uma leitura mais direta — PBIA vs. os 4 polos combinados

O gráfico anterior sobrepõe 5 marcas por termo (a barra do PBIA + um
marcador por polo) e, em termos como *artifici* ou *sustain*, os quatro
marcadores de polo ficam próximos ou sobrepostos — dificultando a
leitura rápida da magnitude. Uma comparação mais direta agrega os
quatro polos numa única barra de referência — a mesma frequência
agregada (`pbia_rest_counts` / `pbia_rest_total`) já usada para
selecionar estes 15 termos por G² na célula acima, não um novo
cálculo — e mostra, lado a lado, apenas **duas barras por termo**:
quanto o PBIA usa cada termo vs. quanto os quatro polos, somados,
usam o mesmo termo.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

rows = top_pbia_terms[::-1]  # menor G² embaixo, maior no topo -- mesma convenção do gráfico anterior
labels = [STEM_DISPLAY_LABEL.get(r[0], r[0]) for r in rows]
pbia_rates = [r[4] for r in rows]
rest_rates = [r[5] for r in rows]  # já é a frequência do PBIA_rest (4 polos combinados), Seção 10.1
y_pos = np.arange(len(rows))
bar_h = 0.36

REST_COLOR = "#4d4b46"  # cinza neutro -- agregado dos 4 polos, não é a identidade de nenhum documento

ax.barh(y_pos + bar_h / 2, pbia_rates, height=bar_h, color=COLOR["PBIA.json"], zorder=3,
        label=LABELS["PBIA.json"])
ax.barh(y_pos - bar_h / 2, rest_rates, height=bar_h, color=REST_COLOR, zorder=3,
        label="4 polos combinados (UE + China + EUA + OCDE)")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Frequência relativa (‰)", color=INK_MUTED, fontsize=9.5)
ax.set_title(f"Os {N_PBIA} termos específicos do PBIA — PBIA vs. os 4 polos combinados",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", pad=12)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(colors=INK_MUTED)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9.5)

fig.tight_layout()
plt.show()


---
# 11. O vocabulário de cada bloco — e por que cada documento está onde está

## UE — AI Continent Action Plan (X=+59,7 · Y=-6,2) e Apply AI Strategy (X=+48,5 · Y=-5,3)

O vocabulário mais forte do polo UE (Seção 5) é **sector** (172 ocorrências
no polo, 16,83‰; G²=194,6 — o termo mais distintivo de qualquer um dos
quatro polos), **strateg[y/-ic]** (129×, 12,62‰), **gigafactory** (36×,
exclusivo do polo — 0 ocorrências no resto), **factory** (60×), **member**
[Estado-membro] (39×), **act** [como em "AI Act"] (59×), **single**
[mercado único] e **uptake** [adoção de IA] (19× cada, também exclusivos).
É um vocabulário de **arquitetura regulatório-institucional multinacional**:
setores econômicos regulados, Estados-membros, atos legislativos da própria
UE, infraestrutura física de computação em grande escala batizada com um
nome de programa específico (*gigafactory*).

Os dois documentos do bloco **não são idênticos entre si** — e a diferença
é ela própria informativa: o **AI Continent Action Plan** concentra-se no
vocabulário de **infraestrutura física** — *factory* (9,43‰) e *gigafactory*
(5,80‰) muito mais presentes que na Apply AI Strategy (1,70‰ e 0,85‰) — ,
enquanto a **Apply AI Strategy** concentra-se no vocabulário de **aplicação
setorial** — *sector* é quase o dobro (20,82‰ vs. 13,42‰) e *uptake*
aparece 4,4× mais (3,19‰ vs. 0,73‰). São duas frentes do mesmo bloco:
construir a infraestrutura (Continent) e fazer os setores econômicos
adotarem-na (Apply) — o que explica por que ambos caem no mesmo quadrante
(X positivo, Y levemente negativo) sem serem o mesmo ponto.

## China — China New Gen (X=-158,9 · Y=-1,3) e AI+ (X=-149,3 · Y=-1,8)

O vocabulário mais forte do polo China é **intelligen[t/-ce]** (254×,
25,69‰; G²=316,4 — o maior G² de todo o corpus, quase o dobro do segundo
colocado), **smart** (103×, exclusivo em relação ao resto: 0,21‰ vs. 10,42‰
no polo), **technolog[y]** (208×), **theory** (61×, quase exclusivo),
**platform** (72×), **sensing** e **method** (exclusivos, 0 ocorrências no
resto). É um vocabulário de **arquitetura técnica de sistemas de IA** —
inteligência, sistemas "inteligentizados", plataformas, sensoriamento —
mais próximo do registro de engenharia do que do registro regulatório da UE
ou geopolítico dos EUA.

Os dois documentos do bloco também divergem de forma legível: o **China New
Gen (2017)** tem mais peso em **smart** (11,31‰ vs. 7,28‰ no AI+) e sobretudo
em **theory** (7,93‰ vs. 0‰ no AI+) — reflexo de ser o documento fundacional,
que enquadra a IA em termos de estágios teóricos de desenvolvimento
científico. Já o **AI+ (2023)** tem muito mais peso em verbos diretivos de
campanha — **promot[e/-ion]** (18,21‰ vs. 9,75‰) e **strength[en]** (18,21‰
vs. 8,84‰) — o registro de "opinião" governamental chinesa, que instrui a
"promover" e "fortalecer" ações, mais do que descreve teoria. Mesmo com essa
diferença de registro e de seis anos de distância, os dois caem quase no
mesmo ponto do eixo X — o vocabulário técnico-de-sistema (*intelligen*,
*technolog*, *develop*) domina o suficiente em ambos para produzir um
polo coeso.

## EUA — America's AI Action Plan (X=-21,2 · Y=+66,7)

O vocabulário mais forte do polo EUA é **feder[al]** (29×, exclusivo — 0
ocorrências no resto dos três polos), **agency** (22×), **adversary** (10×,
exclusivo), **export** (10×, exclusivo), **prioritize** (8×, exclusivo),
**semiconductor**, **convene**, **ally**, **dod** [Department of Defense] —
todos exclusivos ou quase. É um vocabulário de **competição geopolítica e
administração federal**: agências federais mobilizadas, exportação de
semicondutores controlada, adversários e aliados nomeados — o único dos
quatro polos cujo vocabulário mais forte é sobre **quem é competidor e quem
é parceiro**, não sobre o que a tecnologia faz.

**Por que X é levemente negativo (-21,2), não perto de zero?** Os oito
termos do polo China mais usados pelo America's AI Action Plan são
justamente os **menos exclusivos** da lista chinesa — *develop*
(10,99‰), *technolog[y]* (7,96‰), *new* (7,58‰), *intelligen[t/-ce]*
(4,17‰) — vocabulário genérico de desenvolvimento tecnológico que qualquer
plano de IA usa, e que só entrou na lista do polo China porque o corpus
chinês (9.888 tokens) é grande o bastante para tornar até uma vantagem
moderada estatisticamente significativa (Seção 5). Nos termos
**verdadeiramente exclusivos** do polo China — *smart*, *platform*,
*method*, *sensing* — o America's AI Action Plan tem **zero** ocorrências.
Ou seja: o X levemente negativo não significa "o plano americano soa como a
China" — significa que ele não usa o vocabulário mais específico de
nenhum dos dois lados do eixo X, e por isso fica perto do centro nesse
eixo, com uma inclinação estatística modesta e não substantiva para o lado
China. No eixo Y, em contraste, o sinal é forte e inequívoco: 76,92‰ de
densidade do vocabulário EUA (federal/geopolítico) contra apenas 5,59‰ do
vocabulário OCDE.

## OCDE — âncora do polo Y negativo (X=-23,2 · Y=-147,8)

O vocabulário mais forte da OCDE é **trust** (30×, 23,96‰; G²=97,2),
**co-operation** e **stewardship** (10× cada, ambos exclusivos),
**recommendation** (11×), **respons[ible/-ility]** (15×), **principle**
(13×), **appropriate** (11×), **actor** (12×), **lifecycle** (9×),
**stakeholder** (14×), **recognising** (grafia britânica, 5×, exclusivo),
**internation[al]** (16×). É o vocabulário de um **instrumento
jurídico multilateral de soft law** — atores, ciclo de vida do sistema,
princípios, cooperação internacional, tutela ("stewardship") — o mais
distante, dos quatro polos, do registro de "plano de ação nacional" que
caracteriza os outros três.

A posição em Y (-147,8) é **por construção**: como a OCDE é o único
documento que define esse polo, é matematicamente esperado que ela pontue
muito acima de qualquer outro documento nesse vocabulário — por isso ela
aparece no gráfico como âncora de referência (losango vazado), não como um
documento posicionado em pé de igualdade com os outros seis. Já a posição em
**X (-23,2) não é por construção** — a OCDE não define nenhum polo do eixo
X — e ainda assim ela também pontua discretamente do lado China, pelo
mesmo mecanismo do America's AI Action Plan: seu uso de **develop**
(15,97‰) e **promot[e/-ion]** (7,19‰), termos genéricos que entraram na
lista do polo China por efeito de tamanho de amostra, não por afinidade
real com o registro técnico chinês (**smart**, **theory**, **sensing**:
0‰ na OCDE).

---
# 12. Por que o PBIA está posicionado onde está

**Coordenadas do PBIA: X=-38,2 · Y=+6,5** — perto da origem nos dois eixos,
com magnitude pequena em ambos frente aos documentos que de fato definem
cada polo (China: -149 a -159; UE: +48 a +60; EUA: +66,7; OCDE: -147,8).

## O padrão é o mesmo nos quatro polos: camada genérica presente, vocabulário exclusivo ausente

| Polo | Termos genéricos do polo usados pelo PBIA (taxa no PBIA) | Termos exclusivos do polo — PBIA=0 |
|---|---|---|
| **UE** | solution 6,35‰ (**acima** da taxa do próprio polo UE, 4,99‰); sector 5,42‰; action 4,63‰; strateg 4,50‰ | gigafactory, factory, centre, single (mercado único), uptake |
| **China** | develop 20,23‰ (perto da taxa do próprio polo, 24,27‰); technolog 13,75‰; intelligen 7,54‰; new 5,55‰; promot 5,16‰ | smart, theory, sensing; method quase zero (0,13‰) |
| **EUA** | program 8,20‰ (**acima** da taxa do próprio polo EUA, 6,06‰); center 3,97‰; secur[ity] 3,04‰; feder[al] 3,04‰ | adversary, export, prioritize, semiconductor, convene, dod |
| **OCDE** | internation[al] 3,31‰; respons[ible] 3,44‰; right 2,25‰; policy 2,12‰ | co-operation, stewardship, lifecycle, recognising |

Em nenhum dos quatro polos o PBIA está em silêncio — ele usa a camada de
vocabulário **compartilhada** de cada um (a mesma lógica da Seção 6: termos
como *program*, *solution*, *technology*, *international* atravessam
blocos) e, em dois casos (*solution* na UE; *program* nos EUA), usa esse
termo genérico numa taxa até **maior** que a do próprio polo de origem. O
que falta sistematicamente é o vocabulário **mais idiossincrático e
institucionalmente específico** de cada bloco — os nomes de programas
(*gigafactory*), os termos de arquitetura técnica (*smart*, *sensing*), o
vocabulário de competição geopolítica (*adversary*, *semiconductor*) e o
registro de instrumento jurídico internacional (*stewardship*,
*co-operation*). É essa ausência seletiva, não a ausência de vocabulário em
geral, que empurra o PBIA para perto do centro do gráfico nos dois eixos.

## Um contraponto necessário: o centro não é silêncio, é uma voz própria em outro eixo

Este gráfico mede o PBIA **contra vocabulários de fora** — os quatro polos
foram definidos a partir dos outros documentos, nunca do PBIA. A Seção 5 de
`Estudo Gráfico.ipynb` mostrou, por um método independente (G² do PBIA
contra o pool dos outros sete documentos), que o PBIA **tem**, sim, um
vocabulário fortemente distintivo — só que ele está em registro **diferente**
dos quatro eixos aqui construídos: *program* (62× vs. 23× no pool),
*reduction* (35× vs. 1×), *health* (52× vs. 23×), *qualif[ication]* (23× vs.
0×), *structuring* (20× vs. 0×), *evasion* (12× vs. 0×) — vocabulário de
**entrega social concreta** (redução de desigualdade, evasão escolar,
qualificação profissional, saúde pública), que não é o vocabulário
específico de nenhum dos quatro polos UE/China/EUA/OCDE deste gráfico. A
posição central do PBIA nesta bússola, portanto, não deve ser lida como "o
PBIA não tem voz própria" — deve ser lida como "medido nos quatro eixos que
outros blocos definiram, o PBIA fica perto do meio porque sua voz mais forte
está registrada num quinto eixo que este gráfico, por desenho, não mede".

---
# 13. Análise final do gráfico

## Leitura por quadrante

- **UE** (AI Continent +59,7/-6,2; Apply AI Strategy +48,5/-5,3) ocupa com
  folga a metade positiva do eixo X, com Y perto de zero — um vocabulário
  claramente distinto do China no eixo horizontal, mas sem inclinação forte
  nem para o registro federal-geopolítico dos EUA nem para o de
  instrumento internacional da OCDE no eixo vertical.
- **China** (China New Gen -158,9/-1,3; AI+ -149,3/-1,8) ocupa a metade
  negativa do eixo X com a mesma característica: Y perto de zero. Os dois
  blocos UE e China se diferenciam nitidamente **um do outro** (eixo X), mas
  nenhum dos dois se diferencia fortemente dos polos do eixo Y.
- **America's AI Action Plan** (-21,2/+66,7) é o único, entre os cinco
  documentos internacionais, com Y fortemente deslocado — e, como detalhado
  na Seção 11, seu X discretamente negativo é majoritariamente um efeito de
  vocabulário genérico compartilhado com o polo China, não um sinal de
  proximidade real com o registro técnico chinês.
- **OCDE** (âncora, -23,2/-147,8) espelha o America's AI Action Plan no eixo
  X (perto de zero, levemente negativo, pelo mesmo mecanismo) e ocupa o
  extremo do eixo Y — em parte por construção, já que ela própria define
  esse polo.
- **PBIA** (-38,2/+6,5) é o documento mais perto da origem nos dois eixos
  simultaneamente — nenhum dos outros seis chega perto de zero nos dois
  eixos ao mesmo tempo.

## Uma observação estrutural sobre a própria bússola

Três dos sete pontos do gráfico — America's AI Action Plan, PBIA e a
âncora da OCDE — têm X entre -21 e -43: todos discretamente do lado China,
nenhum deles com magnitude perto dos verdadeiros documentos chineses
(-149 a -159). Na prática, o eixo X **só discrimina fortemente** o par
UE do par China; para os outros três pontos, ele os agrupa num mesmo
"meio-campo" indistinto. O eixo Y tem a mesma característica no sentido
oposto: só a America's AI Action Plan e a âncora da OCDE se afastam de
zero — os outros cinco pontos (os dois documentos da UE, os dois da China e
o PBIA) ficam todos entre -9 e +7. Ou seja, nenhum dos dois eixos desta
bússola é, de fato, um contínuo suave entre os sete pontos — cada eixo
separa bem exatamente o par de documentos que o define e deixa os demais
próximos do centro. Isso não invalida o gráfico (é esperado, dado que cada
polo foi definido a partir de vocabulário **específico** de apenas um a
dois documentos), mas significa que a posição central do PBIA no eixo X não
é exclusiva dele — é compartilhada por America's AI Action Plan e pela
âncora da OCDE, os outros dois pontos também sem vocabulário específico de
nenhum dos dois lados desse eixo. O que distingue de fato o PBIA dos outros
seis pontos é estar perto do centro **nos dois eixos ao mesmo tempo** — o
que é a própria evidência textual e numérica reunida nas Seções 10 a 12,
não uma leitura visual isolada do ponto do PBIA.

## Limitações declaradas

- **Polos de tamanho desigual.** UE e China somam dois documentos cada;
  EUA e OCDE são, cada um, um único documento menor (Seção 3). O uso de N
  fixo (Seção 5) equaliza quantos termos cada polo contribui ao escore, mas
  não equaliza o **poder estatístico** do teste G² por trás da escolha
  desses termos — os 15 termos da UE/China vêm de uma lista de candidatos
  ~3x maior que a dos EUA/OCDE.
- **G² compara cada polo contra os outros três combinados**, não par a
  par — identifica o que é distintivo de um polo *no agregado*, não qual
  polo específico mais se aproxima ou mais diverge de outro em cada termo.
- **O sinal de Y do PBIA não é robusto** à escolha de N (Seção 8) — deve
  ser lido como "perto de zero, sem inclinação clara", nunca como "o PBIA
  pende para o lado dos EUA" ou "para o lado da OCDE".
- **Efeito de tradução não controlado.** PBIA e os dois documentos chineses
  chegam ao corpus como traduções para o inglês (do português e do
  mandarim, respectivamente), enquanto UE, EUA e OCDE foram redigidos
  originalmente em inglês. Um possível efeito de registro de tradução
  (calques, construções mais diretas) não foi isolado nem descartado como
  explicação parcial de padrões lexicais.
- **Stemmer conservador fora do dicionário curado** (Seção 2) — verbos em
  gerúndio/passado não cobertos por ele permanecem não normalizados,
  subestimando levemente a densidade real em alguns termos.
- **N=15 é uma escolha do pesquisador**, ainda que testada quanto à
  robustez (Seção 8) — outro valor razoável produziria magnitudes
  diferentes, embora o quadrante de cinco dos seis documentos (todos menos
  o PBIA no eixo Y) se mantenha estável entre N=10 e N=25.